In [ ]:
# for plotting

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import colorcet as cc

sns.set()
sns.set_context('poster')
sns.set_style('ticks')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = 'cmr10'
plt.rcParams["mathtext.fontset"] = 'cm'
plt.rcParams["axes.formatter.use_mathtext"] = True

In [ ]:
import numpy as np
import xarray as xr
from datetime import datetime, timedelta
# from scipy import stats
import pickle
import json

In [ ]:
nx, ny, nz, nt = 512, 512, 120, 241
grid_vol = 0.025*0.025*0.025 # km**3
dts = 0.5 # minute
dx = 25 # m
dy = 25 # m
dz = 25 # m
z = np.arange(dz/2, 3000., dz)
t = np.arange(0, nt)*dts # minutes
ti = np.arange(-0.5, nt, 1.)*dts
zi = np.arange(0., 3001., dz)

In [ ]:
with open('hdf5/cloud_nodes.json', 'r') as f:
    cloud_nodes = json.load(f)
with open('hdf5/events.json', 'r') as f:
    all_clouds = json.load(f)
with open('uninterrupted_large_clouds.json', 'r') as f:
    ul_clouds = json.load(f)
ul_clouds_list = list(ul_clouds.keys())
nc = len(ul_clouds_list)
id_list = np.asarray(ul_clouds_list, dtype=np.int32)
print(nc)

In [ ]:
merge_end_clouds = []
for id, info in ul_clouds.items():
    time_dict = {int(k):v for k, v in info.items() if k.isdigit()}
    last_time = time_dict[max(time_dict.keys())]
    if 'merge_end' in last_time:
        # if 'merge' not in last_time:
        merge_end_clouds.append(id) 
me_id_list = np.asarray(merge_end_clouds, dtype=np.int32)
split_start_clouds = []
for id, info in ul_clouds.items():
    time_dict = {int(k):v for k, v in info.items() if k.isdigit()}
    first_time = time_dict[min(time_dict.keys())]
    if 'split_start' in first_time:
       split_start_clouds.append(id) 
ss_id_list = np.asarray(split_start_clouds, dtype=np.int32)
ss_me_id_list = np.asarray(list(set(split_start_clouds+merge_end_clouds)), dtype=np.int32)
print(me_id_list.shape, ss_id_list.shape, ss_me_id_list.shape)

In [ ]:
with open(f'pkl/core_all_af.pkl', 'rb') as f:
    core_areas = pickle.load(f)
with open(f'pkl/cloud_all_af.pkl', 'rb') as f:
    cloud_areas = pickle.load(f)
with open(f'pkl/plume_all_af.pkl', 'rb') as f:
    plume_areas = pickle.load(f)

In [ ]:
cloud_volumes = cloud_areas.sum(axis=2)
print(cloud_volumes.shape)
cloud_times = (cloud_volumes > 0).sum(axis=1)
plume_volumes = plume_areas.sum(axis=2)
print(plume_volumes.shape)
plume_times = (cloud_volumes > 0).sum(axis=1)

In [ ]:
cbl_c = np.argmax(cloud_areas > 0, axis=2)
ctl_c = nz - 1 -  np.argmax(cloud_areas[:,:,::-1] > 0, axis=2)

In [ ]:
mcbl_c = np.zeros(nc, dtype=np.int32)
mctl_c = np.zeros(nc, dtype=np.int32)
for i in range(nc):
    a = cloud_areas[i,:,:].sum(axis=0)
    mcbl_c[i] = ((np.where(a != 0))[0]).min()
    mctl_c[i] = ((np.where(a != 0))[0]).max()

In [ ]:
with open(f'pkl/qclm.pkl', 'rb') as f:
    qclm = np.asarray(pickle.load(f))
with open(f'pkl/ws.pkl', 'rb') as f:
    ws = np.asarray(pickle.load(f))
with open(f'pkl/qts.pkl', 'rb') as f:
    qts = np.asarray(pickle.load(f))

In [ ]:
mcbl = np.argmax(qclm > 1.0e-6)
print(z[mcbl])
print(mcbl)

In [ ]:
attached_c = (cloud_volumes > 0) & (cbl_c - 5 < mcbl )
print(attached_c.shape)
attached = attached_c.sum(axis=1) > 0
attached_clouds = id_list[attached]
print(attached_clouds.shape)
with open('pkl/attached_clouds.pkl', 'wb') as f:
    pickle.dump(attached_clouds, f)
attached_ind = np.asarray(np.nonzero(np.isin(id_list, attached_clouds, assume_unique=True))[0])

In [ ]:
pure_attached_ind = attached_ind[np.isin(attached_clouds, ss_me_id_list, assume_unique=True, invert=True)]
pure_attached_ind.shape

In [ ]:
with open(f'pkl/core_all_mf.pkl', 'rb') as f:
    core_mfs = pickle.load(f)
with open(f'pkl/cloud_all_mf.pkl', 'rb') as f:
    cloud_mfs = pickle.load(f)
with open(f'pkl/plume_all_mf.pkl', 'rb') as f:
    plume_mfs = pickle.load(f)
core_mfs = core_mfs*dx*dy # now in kg/s unit
cloud_mfs = cloud_mfs*dx*dy # now in kg/s unit
plume_mfs = plume_mfs*dx*dy # now in kg/s unit

In [ ]:
def calc_mean_mf(plume_mfs, mcbl, attached_c):
    nc, nt, nz = plume_mfs.shape
    mf = np.zeros(nc, dtype=float)
    mf_t = np.zeros((nc, nt), dtype=float)
    for i in range(nc):
        a = attached_c[i,:]
        for t in range(nt):
            mf_t[i,t] = plume_mfs[i, t, mcbl-1:mcbl+2].mean()
        if a.sum() != 0:
            mf[i] = (a * mf_t[i,:]).sum()/a.sum()
    return mf
mean_cb_mf = calc_mean_mf(plume_mfs, mcbl, attached_c)

In [ ]:
print(mean_cb_mf.min())
print(mean_cb_mf.max())

In [ ]:
fig = plt.figure(figsize=(12,7))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
_ = ax.hist([z[mcbl_c], z[mcbl_c[pure_attached_ind]]], bins=10, log=True, color=['black', 'red'])
ax.axvline(np.median(z[mcbl_c]), color='black')
ax.axvline(np.median(z[mcbl_c[pure_attached_ind]]), color='red')
ax.legend(['fully tracked', 'fully tracked attached'])
ax.set_xlabel(f'cloud base height (m)')
plt.show()

In [ ]:
property = 'cloud_volume'
x = np.asarray([v[property]*grid_vol for k, v in ul_clouds.items()])
fig = plt.figure(figsize=(12,7))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
_ = ax.hist([x, x[attached_ind]], bins=10, log=True)
ax.legend(['fully tracked', 'fully tracked attached'])
ax.set_xlabel(f'Cloud Volume ($km^3$)')
plt.show()

In [ ]:
property = 'condense_time'
bins = np.arange(0., 101., 10.)
bins[1] = 9.9
# print(bins)
x = np.asarray([v[property]*dts for k, v in ul_clouds.items()])
fig = plt.figure(figsize=(12,7))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
_, cloud_time_bins, _ = ax.hist([x, x[attached_ind]], bins=bins, log=True)
ax.legend(['fully tracked', 'fully tracked attached'])
ax.set_xlabel(f'Cloud Lifetime (minutes)')
plt.show()

In [ ]:
print(f"corrcoef(<M_b>, cloud lifetime) = {np.corrcoef(mean_cb_mf[attached_ind], cloud_times[attached_ind])[0,1]}")
print(f"corrcoef(<M_b>, cloud lifetime) excluding merge_end & split_start = {np.corrcoef(mean_cb_mf[pure_attached_ind], cloud_times[pure_attached_ind])[0,1]}")
print(f"corrcoef(<M_b>, cloud top height) = {np.corrcoef(mean_cb_mf[attached_ind], z[mctl_c[attached_ind]])[0,1]}") 
print(f"corrcoef(<M_b>, cloud top height) excluding merge_end & split_start = {np.corrcoef(mean_cb_mf[pure_attached_ind], z[mctl_c[pure_attached_ind]])[0,1]}") 

In [ ]:
mean_cb_mf_c = np.where(mean_cb_mf > 1.0, mean_cb_mf, 1.1)
clipped_mf = mean_cb_mf_c[pure_attached_ind]

minmf = 0
maxmf = np.log10(np.max(mean_cb_mf) + 10.0)
bins = np.linspace(minmf, maxmf, 10)
nbins = len(bins) - 1
colors = plt.cm.rainbow_r(np.linspace(0, 1, nbins - 3))
full_colors = np.zeros((9, 4), dtype=np.float64)
full_colors[:3, 3] = 1.0
full_colors[3:, :] = colors[::-1, :]

fig = plt.figure(figsize=(12, 7))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
n, bins, patches = ax.hist(
    np.log10(clipped_mf), bins=bins, log=True, histtype="bar", edgecolor="white"
)
ax.set_ylim(1, 10000)
ax.set_xlabel(r"$\log_{10} \langle M_b \rangle$")

for i in range(len(patches)):
    patches[i].set_facecolor(full_colors[i])

n, _ = np.histogram(np.log10(clipped_mf), bins=bins)
sum_t, _ = np.histogram(
    np.log10(clipped_mf), bins=bins, weights=cloud_times[pure_attached_ind] * dts
)
ave_t = np.asarray(sum_t) / np.asarray(n, dtype=float)
bcs = (bins[1:] + bins[:-1]) * 0.5
bin_ids = np.digitize(np.log10(clipped_mf), bins) - 1

fig = plt.figure(figsize=(16, 8))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
colors = plt.cm.rainbow_r(np.linspace(0, 1, nbins - 3))
for i, c in enumerate(colors):
    bn = nbins - i - 1
    cn = pure_attached_ind[bin_ids == bn]
    ax.scatter(
        np.log10(clipped_mf[bin_ids == bn]),
        np.log10(cloud_times[cn] * dts),
        s=5,
        color=c,
    )
    ax.plot(bcs[bn], np.log10(ave_t[bn]), marker="s", markersize=20, color=c)
for i in range(3):
    cn = pure_attached_ind[bin_ids == i]
    ax.scatter(
        np.log10(clipped_mf[bin_ids == i]),
        np.log10(cloud_times[cn] * dts),
        s=5,
        color="black",
    )
    ax.plot(bcs[i], np.log10(ave_t[i]), marker="s", markersize=20, color="black")
ax.set_xlabel(
    r"Lifetime-Mean Plume-Area Cloud-Base Mass-Flux ($kg/s$, log-10-scale)", fontsize=20
)
ax.set_ylabel("Cloud Lifetime (min, log-10-scale)")
plt.show()

n, _ = np.histogram(np.log10(clipped_mf), bins=bins)
sum_z_ctl, _ = np.histogram(
    np.log10(clipped_mf), bins=bins, weights=z[mctl_c[pure_attached_ind]]
)
ave_z_ctl = np.asarray(sum_z_ctl) / np.asarray(n, dtype=float)
# bcs = (bins[1:]+bins[:-1])*0.5

fig = plt.figure(figsize=(16, 8))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
colors = plt.cm.rainbow_r(np.linspace(0, 1, nbins - 3))
for i, c in enumerate(colors):
    bn = nbins - i - 1
    cn = pure_attached_ind[bin_ids == bn]
    ax.scatter(np.log10(clipped_mf[bin_ids == bn]), z[mctl_c[cn]], s=5, color=c)
    ax.plot(bcs[bn], ave_z_ctl[bn], marker="s", markersize=20, color=c)
for i in range(3):
    cn = pure_attached_ind[bin_ids == i]
    ax.scatter(np.log10(clipped_mf[bin_ids == i]), z[mctl_c[cn]], s=5, color="black")
    ax.plot(bcs[i], ave_z_ctl[i], marker="s", markersize=20, color="black")
ax.set_xlabel(
    r"Lifetime-Mean Plume-Area Cloud-Base Mass-Flux ($kg/s$, log-10-scale)", fontsize=20
)
ax.set_ylabel("Cloud Top Height (m)")
ax.set_ylim((500, 2300))
# ax.set_xlim((2, 5.5))
plt.show()

In [ ]:
mean_cb_mf_c = np.where(mean_cb_mf > 1.0, mean_cb_mf, 1.1)
clipped_mf = mean_cb_mf_c[attached_ind]
ll = len(clipped_mf)
print(ll)

minmf = 0
maxmf = np.log10(np.max(mean_cb_mf) + 10.0)
bins = np.linspace(minmf, maxmf, 10)
nbins = len(bins) - 1
colors = plt.cm.rainbow_r(np.linspace(0, 1, nbins - 3))
full_colors = np.zeros((9, 4), dtype=np.float64)
full_colors[:3, 3] = 1.0
full_colors[3:, :] = colors[::-1, :]

weights = np.zeros(ll) + 1.0/float(ll)
fig = plt.figure(figsize=(12, 7))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
n, bins, patches = ax.hist(
    np.log10(clipped_mf), bins=bins, log=True, histtype="bar", edgecolor="white", weights = weights,
)
print(bins)
print(n)
print(sum(n))
with open('binned_cloud_numbers.bomex.pkl', 'wb') as f:
    pickle.dump(n, f)
with open('bins.bomex.pkl', 'wb') as f:
    pickle.dump(bins, f)
ax.set_ylim(0.001, 0.5)
yticks = np.asarray((0.001, 0.01, 0.1, 0.2, 0.3, 0.4))
ax.set_yticks(yticks)
ax.set_yticklabels(("0.1%", "1%", "10%", "20%", "30%", "40%"))
ax.set_xlabel(r"$\log_{10} \langle M_b \rangle$")

for i in range(len(patches)):
    patches[i].set_facecolor(full_colors[i])

n, _ = np.histogram(np.log10(clipped_mf), bins=bins)
sum_t, _ = np.histogram(
    np.log10(clipped_mf), bins=bins, weights=cloud_times[attached_ind] * dts
)
ave_t = np.asarray(sum_t) / np.asarray(n, dtype=float)
bcs = (bins[1:] + bins[:-1]) * 0.5
bin_ids = np.digitize(np.log10(clipped_mf), bins) - 1

fig = plt.figure(figsize=(16, 8))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
colors = plt.cm.rainbow_r(np.linspace(0, 1, nbins - 3))
for i, c in enumerate(colors):
    bn = nbins - i - 1
    cn = attached_ind[bin_ids == bn]
    ax.scatter(
        np.log10(clipped_mf[bin_ids == bn]),
        np.log10(cloud_times[cn] * dts),
        s=5,
        color=c,
    )
    ax.plot(bcs[bn], np.log10(ave_t[bn]), marker="s", markersize=20, color=c)
for i in range(3):
    cn = attached_ind[bin_ids == i]
    ax.scatter(
        np.log10(clipped_mf[bin_ids == i]),
        np.log10(cloud_times[cn] * dts),
        s=5,
        color="black",
    )
    ax.plot(bcs[i], np.log10(ave_t[i]), marker="s", markersize=20, color="black")
ax.set_xlabel(
    r"Lifetime-Mean Plume-Area Cloud-Base Mass-Flux ($kg/s$, log-10-scale)", fontsize=20
)
ax.set_ylabel("Cloud Lifetime (min, log-10-scale)")
plt.show()

n, _ = np.histogram(np.log10(clipped_mf), bins=bins)
sum_z_ctl, _ = np.histogram(
    np.log10(clipped_mf), bins=bins, weights=z[mctl_c[attached_ind]]
)
ave_z_ctl = np.asarray(sum_z_ctl) / np.asarray(n, dtype=float)
# bcs = (bins[1:]+bins[:-1])*0.5

fig = plt.figure(figsize=(16, 8))
ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
colors = plt.cm.rainbow_r(np.linspace(0, 1, nbins - 3))
for i, c in enumerate(colors):
    bn = nbins - i - 1
    cn = attached_ind[bin_ids == bn]
    ax.scatter(np.log10(clipped_mf[bin_ids == bn]), z[mctl_c[cn]], s=5, color=c)
    ax.plot(bcs[bn], ave_z_ctl[bn], marker="s", markersize=20, color=c)
for i in range(3):
    cn = attached_ind[bin_ids == i]
    ax.scatter(np.log10(clipped_mf[bin_ids == i]), z[mctl_c[cn]], s=5, color="black")
    ax.plot(bcs[i], ave_z_ctl[i], marker="s", markersize=20, color="black")
ax.set_xlabel(
    r"Lifetime-Mean Plume-Area Cloud-Base Mass-Flux ($kg/s$, log-10-scale)", fontsize=20
)
ax.set_ylabel("Cloud Top Height (m)")
ax.set_ylim((500, 2300))
# ax.set_xlim((2, 5.5))
plt.show()